In [0]:
s3_path = "s3a://ecommerce-dataanalysis"
customer_df = spark.read.csv(f"{s3_path}/olist_customers_dataset.csv", header=True, inferSchema=True)
geolocation_df = spark.read.csv(f"{s3_path}/olist_geolocation_dataset.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv(f"{s3_path}/olist_order_items_dataset.csv", header=True, inferSchema=True)
order_payment_df = spark.read.csv(f"{s3_path}/olist_order_payments_dataset.csv", header=True, inferSchema=True)
order_reviews_df = spark.read.csv(f"{s3_path}/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
orders_df = spark.read.csv(f"{s3_path}/olist_orders_dataset.csv", header=True, inferSchema=True)
products_df = spark.read.csv(f"{s3_path}/olist_products_dataset.csv", header=True, inferSchema=True)
sellers_df = spark.read.csv(f"{s3_path}/olist_sellers_dataset.csv", header=True, inferSchema=True)
product_category_name_df = spark.read.csv(f"{s3_path}/product_category_name_translation.csv", header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import *
customer_df.printSchema()

# Find the missing values in the DataSet

In [0]:
def missing_values(df , name):
    print(f"find the missing value in the : {name}")
    df.select([count(when(col(c).isNull() , 1)).alias(c) for c in df.columns]).show()

In [0]:
missing_values(customer_df , 'customer_df')

In [0]:
missing_values(geolocation_df , 'customer_df')

In [0]:
missing_values(orders_df , 'customer_df')

In [0]:
missing_values(order_items_df , 'order_items_df')

In [0]:
missing_values(order_payment_df , 'order_payments_df')

In [0]:
missing_values(order_reviews_df , 'order_reviews_df')

In [0]:
missing_values(products_df , 'products_df')

In [0]:
missing_values(sellers_df , 'sellers_df')

In [0]:
missing_values(product_category_name_df , 'product_category_name_df')

In [0]:
customer_df.printSchema()
cleaned_customer_df = customer_df.dropna(subset=['customer_id'] , how='all')

In [0]:
cleaned_customer_df.count()

In [0]:
missing_values(geolocation_df , 'geolocation_df')

In [0]:
missing_values(orders_df , 'orders')

In [0]:
cleaned_orders_df = orders_df.dropna(subset=['order_id' , 'customer_id' , 'order_status'] , how='all')

In [0]:
missing_values(cleaned_orders_df , 'cleaned_orders_df')

In [0]:
cleaned_orders_df = cleaned_orders_df.fillna({'order_approved_at' : '9999-01-01 00:00:00','order_delivered_carrier_date' : '9999-01-01 00:00:00','order_delivered_customer_date' : '9999-01-01 00:00:00'})

In [0]:
missing_values(cleaned_orders_df , 'cleaned_orders_df')

In [0]:
missing_values(order_items_df , 'order_items_df')

In [0]:
missing_values(order_payment_df , 'order_payment')

In [0]:
cleaned_customer_df.printSchema()

In [0]:
cleaned_customer_df = cleaned_customer_df.dropDuplicates(['customer_id'])

In [0]:
order_payment_df.show(10)

In [0]:
missing_values(order_reviews_df , 'order_reviews')

In [0]:
cleaned_order_reviews_df = order_reviews_df.dropna(subset=['review_id' , 'order_id' , 'review_score'])

In [0]:
missing_values(cleaned_order_reviews_df , 'cleaned_order_reviews_df')

In [0]:
order_payment_df.show(10)
order_payment_df.groupBy('order_id').count().orderBy('count' , ascending=False).show(10)

In [0]:
from pyspark.sql.functions import when
cleaned_order_payment_df = order_payment_df.withColumn('payment_type' , when(col('payment_type')=='credit_card' , 'Credit Card')\
                                                                          .when(col('payment_type')=='debit_card' , 'Debit Card')\
                                                                          .when(col('payment_type')=='boleto' , 'Bank Transfer')\
                                                                          .otherwise(col('payment_type')))

In [0]:
cleaned_order_payment_df.printSchema()

In [0]:
cleaned_order_reviews_df.printSchema()

In [0]:
from pyspark.ml.feature import Imputer

In [0]:
imputer = Imputer().setInputCols(['review_score']).setOutputCols(['review_score_imput']).setStrategy('median')
cleaned_order_reviews_df = imputer.fit(cleaned_order_reviews_df).transform(cleaned_order_reviews_df)

In [0]:
missing_values(order_reviews_df , 'order_review')

In [0]:
missing_values(cleaned_order_reviews_df , 'cleaned_order_reviews_df')

In [0]:
cleaned_order_reviews_df.show(5)

In [0]:
cleaned_order_payment_df = cleaned_order_payment_df.drop('tempered_payment_value' , 'tempered_payment_value_imput')

In [0]:
cleaned_order_payment_df.show(10)
cleaned_order_payment_df.printSchema()

In [0]:
cleaned_order_payment_df = cleaned_order_payment_df.withColumns({'payment_sequential':col('payment_sequential').cast('string'),'payment_installments' : col('payment_installments').cast('string')})

In [0]:
cleaned_order_payment_df.printSchema()

In [0]:
cleaned_orders_df.show(10)
order_items_df.show(5)

In [0]:
order_items_df.select('price').summary().show()

In [0]:
cleaned_products_df = products_df.withColumn('product_size' , when(col('product_weight_g') < 500 , 'small').when((col('product_weight_g') >= 500) & (col('product_weight_g') < 2000), 'mdeium').otherwise('large'))

In [0]:
cleaned_products_df.show(10)

# Remove the Outliers

In [0]:
quantiles = order_items_df.approxQuantile('price' , [0.01 , 0.99] , 0)
low_cutoff, high_cutoff=quantiles[0],quantiles[1] 

In [0]:
cleaned_order_items = order_items_df.filter((col('price') >= low_cutoff)  & (col('price') <= high_cutoff))

In [0]:
cleaned_order_items.select('price').summary().show()

In [0]:
cleaned_order_payment_df.summary().show()

In [0]:
# cleaned_order_payment_df = cleaned_order_payment_df.approxQuantile('')
cleaned_order_items.printSchema()

In [0]:
order_with_detail = cleaned_orders_df.join(cleaned_order_items , 'order_id' , 'left')\
                                     .join(cleaned_order_payment_df , 'order_id' , 'left')\
                                     .join(cleaned_customer_df , 'customer_id' , 'left')

In [0]:
order_with_detail = order_with_detail.withColumn('delivery_time' , datediff(col('order_delivered_customer_date') , col('order_purchase_timestamp')))

In [0]:
order_with_detail.show(5)

In [0]:
# order_with_detail.write.mode('overwrite').parquet('/data/')
order_with_detail = spark.read.format('parquet').load('/data/olist_proc/orderwithdetail.parquet')